In [1]:
import psycopg
print("psycopg version:", psycopg.__version__)

psycopg version: 3.3.3


In [5]:
# Cell 3 — Verify ipd3 tables were created
import psycopg
import pandas as pd

conn = psycopg.connect(host='platinum', dbname='forge')

df_tables = pd.read_sql("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'ipd3'
    ORDER BY table_name
""", conn)

print("Tables and views in ipd3 schema:")
print(df_tables.to_string(index=False))

conn.close()

Tables and views in ipd3 schema:
  table_name table_type
    episodes BASE TABLE
  llm_agents BASE TABLE
research_log BASE TABLE
     results BASE TABLE
      rounds BASE TABLE


/tmp/ipykernel_2693363/4085925687.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql("""


In [6]:
# Cell 4 — Load all 91 JSON result files into ipd3 schema
import sys, os

# Make sure forgedb.py is importable
sys.path.insert(0, '/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3')
os.chdir('/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3')

from forgedb import ForgeDB

db = ForgeDB(host='platinum', dbname='forge')   # user = rayudu (auto-detected)

RESULTS_DIR = './results/'

print(f"Loading JSON files from: {os.path.abspath(RESULTS_DIR)}")
print(f"Files found: {len([f for f in os.listdir(RESULTS_DIR) if f.endswith('.json')])}")
print()

results = db.load_batch(RESULTS_DIR, pattern='*.json', user_name='rayudu')

print(f"\n--- Load Summary ---")
print(f"Loaded:  {len(results['loaded'])}")
print(f"Skipped: {len(results['skipped'])}  (already in DB — safe)")
print(f"Failed:  {len(results['failed'])}")

if results['failed']:
    print("\nFailed files:")
    for fp, err in results['failed']:
        print(f"  {os.path.basename(fp)}: {err[:80]}")

db.close()

Loading JSON files from: /home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/results
Files found: 91


--- Load Summary ---
Loaded:  91
Skipped: 0  (already in DB — safe)
Failed:  0


In [7]:
# Cell 5 — Check row counts in all ipd3 tables
import psycopg
import pandas as pd

conn = psycopg.connect(host='platinum', dbname='forge')

counts = pd.read_sql("""
    SELECT 'results'    AS table_name, COUNT(*) AS row_count FROM ipd3.results
    UNION ALL
    SELECT 'llm_agents',               COUNT(*)              FROM ipd3.llm_agents
    UNION ALL
    SELECT 'episodes',                 COUNT(*)              FROM ipd3.episodes
    UNION ALL
    SELECT 'rounds',                   COUNT(*)              FROM ipd3.rounds
    UNION ALL
    SELECT 'research_log',             COUNT(*)              FROM ipd3.research_log
""", conn)

conn.close()
print(counts.to_string(index=False))

  table_name  row_count
     results         91
  llm_agents        273
    episodes      13245
      rounds     264900
research_log          0


/tmp/ipykernel_2693363/2416965977.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  counts = pd.read_sql("""


In [11]:
# Cell 6 — Experiment summary: one row per run, all 3 agents as columns
import psycopg
import pandas as pd

conn = psycopg.connect(host='platinum', dbname='forge')

df_summary = pd.read_sql("""
    SELECT
        filename,
        comment,
        cfg_temperature         AS temp,
        cfg_history_window_size AS hw,
        cfg_reset_between_episodes AS reset,
        agent_0_model,
        ROUND(agent_0_cooperation_rate::NUMERIC, 3) AS a0_coop,
        agent_1_model,
        ROUND(agent_1_cooperation_rate::NUMERIC, 3) AS a1_coop,
        agent_2_model,
        ROUND(agent_2_cooperation_rate::NUMERIC, 3) AS a2_coop,
        ROUND(((agent_0_cooperation_rate +
                agent_1_cooperation_rate +
                agent_2_cooperation_rate)/3)::NUMERIC, 3) AS mean_coop
    FROM ipd3.experiment_summary_vw
    ORDER BY timestamp
""", conn)

conn.close()
print(f"Shape: {df_summary.shape}")
df_summary.head(10)

Shape: (91, 12)


/tmp/ipykernel_2693363/372044083.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_summary = pd.read_sql("""


,filename,comment,temp,hw,reset,agent_0_model,a0_coop,agent_1_model,a1_coop,agent_2_model,a2_coop,mean_coop
0,3agent_game_20260325_015733.json,baseline run,0.7,10,True,llama3:8b-instruct-q5_K_M,0.860,llama3:8b-instruct-q5_K_M,0.910,llama3:8b-instruct-q5_K_M,0.900,0.890
1,3agent_game_20260325_231853.json,baseline run,0.7,10,True,llama3:8b-instruct-q5_K_M,0.630,llama3:8b-instruct-q5_K_M,0.630,llama3:8b-instruct-q5_K_M,0.590,0.617
2,3agent_game_20260325_234255.json,baseline run,0.7,10,True,llama3:8b-instruct-q5_K_M,0.840,llama3:8b-instruct-q5_K_M,0.760,llama3:8b-instruct-q5_K_M,0.760,0.787
3,3agent_game_20260330_214736.json,baseline run,0.7,10,True,llama3:8b-instruct-q5_K_M,0.694,llama3:8b-instruct-q5_K_M,0.726,llama3:8b-instruct-q5_K_M,0.740,0.720
4,3agent_game_20260331_000658.json,baseline run,0.7,10,True,llama3:8b-instruct-q5_K_M,0.742,llama3:8b-instruct-q5_K_M,0.753,llama3:8b-instruct-q5_K_M,0.734,0.743
5,3agent_game_20260331_070629.json,baseline run,0.2,10,True,llama3:8b-instruct-q5_K_M,0.779,llama3:8b-instruct-q5_K_M,0.782,llama3:8b-instruct-q5_K_M,0.788,0.783
6,3agent_game_20260331_122730.json,baseline run,0.2,10,True,llama3:8b-instruct-q5_K_M,0.855,llama3:8b-instruct-q5_K_M,0.885,llama3:8b-instruct-q5_K_M,0.867,0.869
7,3agent_game_20260331_125226.json,W5_T02_ResetTrue,0.2,5,True,llama3:8b-instruct-q5_K_M,0.766,llama3:8b-instruct-q5_K_M,0.773,llama3:8b-instruct-q5_K_M,0.771,0.770
8,3agent_game_20260331_160310.json,W20_T02_ResetTrue,0.2,20,True,llama3:8b-instruct-q5_K_M,0.829,llama3:8b-instruct-q5_K_M,0.817,llama3:8b-instruct-q5_K_M,0.814,0.820
9,3agent_game_20260331_165003.json,W5_T07_ResetTrue,0.7,5,True,llama3:8b-instruct-q5_K_M,0.730,llama3:8b-instruct-q5_K_M,0.782,llama3:8b-instruct-q5_K_M,0.750,0.754


In [ ]:
# Cell 8 — Using ForgeDB class methods directly
import sys
sys.path.insert(0, '/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3')
from forgedb import ForgeDB

db = ForgeDB(host='platinum', dbname='forge')

# Equivalent to enriched_registry.csv (one row per run)
df_runs = db.get_summary()
print(f"Runs:    {df_runs.shape}")

# Equivalent to episode_level.csv (one row per episode per run)
df_eps = db.get_episode_summary()
print(f"Episodes: {df_eps.shape}")

# Filter: only runs with comment containing '3L'
df_3l = db.get_summary(comment='%3L%')
print(f"3L runs: {len(df_3l)}")

# Filter: runs between two dates
df_april = db.get_episode_summary(
    start_date='2026-04-01',
    end_date='2026-04-30'
)
print(f"April episodes: {len(df_april)}")

# Add a research log entry
db.add_log(
    subject='ipd3 schema load complete',
    remarks='Loaded 91 JSON files. 6 model mixes, 91 runs, ~13650 episode rows, ~273000 round rows.',
    tags=['ipd3', 'load', 'setup']
)

db.close()

In [9]:
# Cell 9 — Mean cooperation rate by model comment (model mix label)
import psycopg
import pandas as pd
import matplotlib.pyplot as plt

conn = psycopg.connect(host='platinum', dbname='forge')

df = pd.read_sql("""
    SELECT
        r.comment,
        r.cfg_temperature                                  AS temp,
        r.cfg_history_window_size                          AS hw,
        ROUND(AVG(a.overall_cooperation_rate)::NUMERIC, 4) AS mean_coop,
        COUNT(DISTINCT r.results_id)                       AS n_runs
    FROM ipd3.results r
    JOIN ipd3.llm_agents a ON a.results_id = r.results_id
    GROUP BY r.comment, r.cfg_temperature, r.cfg_history_window_size
    ORDER BY mean_coop DESC
""", conn)
conn.close()

print(df.to_string(index=False))

                               comment  temp  hw  mean_coop  n_runs
             3Gemma_W10_T1.0_ResetTrue   1.0  10     1.0000       1
              3Gemma_W5_T1.0_ResetTrue   1.0   5     1.0000       1
              3Gemma_W5_T0.7_ResetTrue   0.7   5     1.0000       1
              3Gemma_W5_T0.2_ResetTrue   0.2   5     1.0000       1
             3Gemma_W20_T1.0_ResetTrue   1.0  20     1.0000       1
             3Gemma_W20_T0.7_ResetTrue   0.7  20     1.0000       1
             3Gemma_W20_T0.2_ResetTrue   0.2  20     1.0000       1
             3Gemma_W10_T0.2_ResetTrue   0.2  10     1.0000       1
             3Gemma_W10_T0.7_ResetTrue   0.7  10     1.0000       1
                     W5_T02_ResetFalse   0.2   5     0.9923       1
                    W10_T0.2_ResetTrue   0.2  10     0.9443       1
                    W20_T0.2_ResetTrue   0.2  20     0.9137       1
                    W20_T07_ResetFalse   0.7  20     0.8983       1
      2Gemma_1Llama_W10_T0.2_ResetTrue   0.2  10

/tmp/ipykernel_2693363/2694929001.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


In [12]:
# Cell 8 — Using ForgeDB class methods directly
import sys
sys.path.insert(0, '/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3')

from forgedb import ForgeDB

db = ForgeDB(host='platinum', dbname='forge')

# Equivalent to enriched_registry.csv (one row per run)
df_runs = db.get_summary()
print(f"Runs:    {df_runs.shape}")

# Equivalent to episode_level.csv (one row per episode per run)
df_eps = db.get_episode_summary()
print(f"Episodes: {df_eps.shape}")

# Filter: only runs with comment containing '3L'
df_3l = db.get_summary(comment='%3L%')
print(f"3L runs: {len(df_3l)}")

db.close()

Runs:    (91, 35)
Episodes: (4415, 18)
3L runs: 0


In [13]:
# Cell 10 — Log your analysis session
import sys
sys.path.insert(0, '/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3')

from forgedb import ForgeDB

db = ForgeDB(host='platinum', dbname='forge')

log_id = db.add_log(
    subject='Analysis session — cooperation rates',
    remarks='Verified ipd3 schema load. 91 runs loaded. Ran cooperation rate summary query.',
    tags=['ipd3', 'analysis', 'rayudu']
)

# View the log
df_log = db.get_log()
print(df_log[['log_id', 'log_dttm', 'subject', 'remarks']].to_string(index=False))

db.close()

Added log entry 1: Analysis session — cooperation rates
 log_id                         log_dttm                              subject                                                                        remarks
      1 2026-04-27 21:42:16.311866-06:00 Analysis session — cooperation rates Verified ipd3 schema load. 91 runs loaded. Ran cooperation rate summary query.
